# Procedure for processor integration with static payload

This notebook is the main template for integration tests of DPR processors (new processors and new versions of existing processors). It is used to run a processor using a static payload inside the RS-Python environment outside of a workflow.   
Use it for the **STEP 2 of DPR integration procedure**: https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/1.+DPR+integration+PROCEDURE   

Along with this notebook, the *config* folders contain the configuration files used for the processors currently integrated, these files are described and refered to in this notebook. Each folder is in the format `config_<PROCESSOR_NAME>` and includes a README file listing the origin of the files it contains.

**IMPORTANT:** Please keep this notebook up-to-date as it will be used throughout the duration of the project.

## Table of Contents

* [**1 - Add new processor**](#step1) - *Mandatory code updates when integrating a new processor*
* [**2 - Retrieve the test data and create the payload**](#step2) - *Creation and configuration of the files needed for the processor to be executed*
* [**3 - Demo initialization and Dask cluster creation**](#step3) - *Creation of the Dask cluster for the processor*
* [**4 - Run the processor**](#step4) - *Execution of the processor*
* [**5 - Retrieve informations and shutdown cluster**](#step5) - *Tips for understanding the processor's execution results*

## 1 - Add new processor <a class="anchor" id="step1"></a>

**Skip this part if you are integrating a new version of an previously integrated processor.**   

If you are integrating a processor that was never previously integrated, first make sure that all the needed Git repositories are updated to support this processor and that an image for it has been generated and is available in the Github artifacts.   

Check the following points:
- In [**rs-workflow-env**](https://github.com/RS-PYTHON/rs-workflow-env), add a requirements file for the new processor in [this folder](https://github.com/RS-PYTHON/rs-workflow-env/tree/develop/docker/eopf/resources) and update the following files taking example on what was done for the existing processors: [*build_dask_eopf.py*](https://github.com/RS-PYTHON/rs-workflow-env/blob/develop/docker/eopf/build_dask_eopf.py) and [*build-dask-eopf.yaml*](https://github.com/RS-PYTHON/rs-workflow-env/blob/develop/.github/workflows/build-dask-eopf.yml).
- After pushing the changes of the previous point to Git, wait for the CI to finish and check that the Dask image for your processor is available here with the name `ghcr.io/rs-python/dask/<YOUR_PROCESSOR>/k8s:<YOUR_BRANCH_NAME>`: https://github.com/orgs/RS-PYTHON/packages. The tag is "latest" only if the changes are in the "develop" branch.
- Update the following files in [**rs-dpr-service**](https://github.com/RS-PYTHON/rs-dpr-service) with the new processor, in the same way it is done for the previous ones: [*eopf_processors.py*](https://github.com/RS-PYTHON/rs-dpr-service/blob/develop/rs_dpr_service/processors/eopf_processors.py), [*main.py*](https://github.com/RS-PYTHON/rs-dpr-service/blob/develop/rs_dpr_service/main.py) and [*geoapi.yaml*](https://github.com/RS-PYTHON/rs-dpr-service/blob/develop/config/geoapi.yaml), and also add an example tasktable in the [*config folder*](https://github.com/RS-PYTHON/rs-dpr-service/tree/develop/config).
- Same in [**rs-client-libraries**](https://github.com/RS-PYTHON/rs-client-libraries): [*dpr_client.py*](https://github.com/RS-PYTHON/rs-client-libraries/blob/develop/rs_client/ogcapi/dpr_client.py) and add the wanted workflows in [*rs_workflows*](https://github.com/RS-PYTHON/rs-client-libraries/tree/develop/rs_workflows).
- Add a notebook in [**rs-demo**](https://github.com/RS-PYTHON/rs-demo) to create a Dask cluster using the image from the second point, in the [*init-dask-clusters* folder](https://github.com/RS-PYTHON/rs-demo/tree/develop/notebooks/init-dask-clusters) (follow what was done for the existing processors). Also update the [**nginx.conf**](https://github.com/RS-PYTHON/rs-demo/blob/develop/local-mode/nginx.conf) file and add a new folder for the files used to run the processor in the same folder as this notebook, named `config_<PROCESSOR_NAME>` where PROCESSOR_NAME is the name given to the processor in the DprProcessor class of *rs-client-libraries* that you updated in the previous point (defined in [*dpr_client.py*](https://github.com/RS-PYTHON/rs-client-libraries/blob/develop/rs_client/ogcapi/dpr_client.py))
- In [**rs-server-deployment**](https://github.com/RS-PYTHON/rs-server-deployment), update the *values.yaml* file of the *rs-dpr-service* configuration corresponding to your processor, usually [this one](https://github.com/RS-PYTHON/rs-server-deployment/blob/develop/apps/rs-dpr-service-py-3-11-7-dask-2024-5-2/values.yaml).

## 2 - Retrieve the test data and create the payload <a class="anchor" id="step2"></a>

This part helps gives details on how to retrieve the correct files and data needed for the integration tests.   

### 2.1 - Identify the correct payload and test data

Along with this notebook, you can find a set of *config* folders, containing the latest version of the config files and payload for each processor. In each one of these folders, use the *readme* file that lists the exact origin of all the files included.
Each processor uses the same files for a basic test run: a **payload**, a variable number of **configuration files** specific to the processor and a **logging configuration file**. This section will help you retrieving the correct files. Please keep all of the files up-to-date when integrating processors, and add the files corresponding to the new processors.   

**payload.yaml:** this is the main set of instructions to run a processor, that identifies which files will be processed. Usually you can get this file from the processor's repository, using a test or an example payload from there. For the L0 processor for example, they can be found in their [validation repository](https://gitlab.eopf.copernicus.eu/L0/validation). See the content of the readme file for existing processors to get the exact location of the payload used. If the processor you are testing is a new version of an already integrated processor, do not directly reuse the payload from the previous version but make sure to compare it with the new versions of the payloads in the repository, sometimes the payload format changes between two versions.

**Processor's configuration files:** these files can vary a lot depending on the processor, some don't have any at all, and for existing processors they are listed in the readme file. However, you can usually get them from the processor's repository, where they have the default values, and use them as is, without any change. Make sure to update them if you are testing a new version of an already integrated processor aswell.   

**logging_config.yaml:** logs configuration. Usually, the one given here can be directly reused, with only changing the output file name with the correct processor's name. In most cases, you can get a logging configuration file from the processor's repository too, but it should be similar to the one here.   

Once you have the necessary files, identify the **test data** needed for your test run. There are two cases here:
1. If you are reusing data from a previous integration test, you don't have to copy it again in most cases (unless the new version you are integrating has specific changes concerning the inputs used, in that case consider you are in the second case). In that case, you can skip the next step because the data is already available.
2. If you are integrating a new processor you need to identify which test data you are going to use before handling it in next step. In that case, the origin of the test data to be used is listed in the original payload you selected previously, each time with a path to a file in a S3 bucket such as `s3://origin_bucket/path/to/test/data`. List all the paths needed for the test run and go to the next step.

**Note:** It is technically possible to use a completely different test data set in case you are familiar with the processor, as long as it's compliant with the inputs expected by the processor. We do not recommend it for a standard integration test however.

### 2.2 - Copy the data and fill the payload

Each processor needs **test data**, that is input files in variable numbers and formats, usually coming from previous processor outputs or CADIP or AUXIP stations in a real case scenario.   
For this integration test, we will manually upload the test data that was selected and identified at previous step, to keep it simple. This section will help you retrieving the correct files and upload them to an S3 bucket to test a DPR processor.   

**NOTE:** If you are working with a new version of an already integrated processor, unless the data needed is different than the one for the previous version, we recommend using the same test data accross each version of a processor. This allows a better evaluation of the evolutions of a processor.   

**Copy the test data:** if you are using the same data as the one used in the validation tests of the processor, the easiest way is to copy it from its origin S3 bucket to the one you are using for the integration tests. You can try [this procedure](https://stackoverflow.com/questions/12700921/s3-moving-files-between-buckets-on-different-accounts) to move data between two unrelated S3 bucket, or follow these instructions to do it manually:
- Get the *endpoint URL* for the S3 bucket originally containing the data. Usually you can find it in the origin repository of the processor (for example, it is available for L0 [here](https://gitlab.eopf.copernicus.eu/L0/validation#s3-bucket-credentials-configuration))
- Get a secret key and an access key to access this bucket. If you don't have any, get help from a CPM administrator who can provide you with one. Use them with the endpoint URL to create a *.s3cfg* file.
- Get the same values for the bucket used for test. If you don't already have them, retrieve them using the OSAM service [on the dev cluster](https://dev-rspy-ovh.esa-copernicus.eu/docs#/OSAM%20service) or [on the OPS cluster](https://rspy.ops.rs-python.eu/docs#/OSAM%20service).
- Copy the data from the source bucket to the test bucket using *s3cmd*. You will need two separate *s3cfg* files to perform this operation, as the source and destination buckets are on different endpoints, and run the following commands: `s3cmd -c <SOURCE_S3CFG_FILE> get s3://bucket/source/path/to/data local/temporary/path [--recursive]` followed by `s3cmd -c <TARGET_S3CFG_FILE> put local/temporary/path s3://bucket/target/path/to/data [--recursive]`

**Stage the test data:** for information, if you have advanced knowledge of the RS-Python systems and the processor, you can also run a regular staging task and use the path of the staged data in the S3 bucket in the payload.    

**Fill the payload:** in the payload retrieved from the processor's repository or adapted from previous tests, adapt the following elements:
- the **data location**: change the value of `path` in each I/O entry of the payload to match with the location of the test data and the expected output
- the **s3 access configuration**: for each product, add a `storage_options` field in `store_params`, with the needed information to access the S3 bucket where the data is (key, secret, ...). In most cases, use environment variables for these values and just copy-paste this block from the example payload. If your environment is correctly configured, you should not have to change anything.
- the **configuration file locations**: following one of the existing payloads, add the `secret`, `logging` and `config` fields at the bottom of the payload with the correct file paths for your test case
- the **Dask configuration** section: if you got the payload from a processor's repository and there is this section, remove it. It is populated automatically by the rs-dpr-service.

Once the data is available and the payload filled, you should be ready to run the processor.

## 3 - Demo initialization and Dask cluster creation <a class="anchor" id="step3"></a>

This part initializes the demo and creates the Dask cluster for the chosen processor.   
You can change the "Optional change" parts as you need, but we do not recommend changing other parts of the code.   

Run all cells in this section.

### 3.1 - Setup variables

Set up the configuration needed to spawn the Dask cluster for a processor.   
Choose the processor you want to test, then set a specific tag for the image to use if you are still in a development phase.   
The other optional parameters are not needed in most of the cases, but they can be changed in the third cell if needed.

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.widget_utils import *

# Choose processor to create a Dask cluster for
dpr_proc_radio

In [ ]:
# "latest" tag is (by default) the latest image released by "rs-workflow-env" repository for this processor.
# If your image is on a development branch, use the associated tag for this image instead of "latest" in the line below.
# Example:
# processor_image_tag = "feat-rspy999-my-dev-branch"

# === OPTIONAL CHANGE ===
# Specify a tag for the image to use
processor_image_tag = "latest"

In [ ]:
# Experimental DPR processor configuration, used only for testing.
# Localcluster is for advanced debugging (local mode) or performance optimization for some cases (cluster mode).
# Local files is a cache system for the files processed.
# Documentation on localcluster and local_files mode, see:
#   - https://github.com/RS-PYTHON/rs-dpr-service/blob/develop/rs_dpr_service/utils/settings.py
#   - https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/1.+Validation+reports

# === OPTIONAL CHANGE ===
# Set the "enabled" field to True to use the localcluster config
# and tune the parameters as you want
experimental_config = {
    "local_cluster": {
        "enabled": False, # Use False to disable
        "n_workers": 4,
        "memory_limit": "58GiB",
    },
    "local_files": {
        "local_dir": None, #"/tmp/data", # Use None to disable
        "overwrite_input": False,
        "upload_output": True,
    },
}

# === OPTIONAL CHANGE ===
# Configuration parameters for the Dask cluster creation (number of nodes and allocated resources)
# The default values for each processor are implemented in their respective notebooks in rs-demo/notebooks/init-dask-clusters. You can overwrite them here for testing.
dask_cluster_parameters = {
    # "scale": 1,
    # "big_resources": True,   # provide more ram and cpu
    # "worker_cores": 4,       # number of CPU per worker 
    # "worker_memory": 58,     # memory per worker in GB
}

# === OPTIONAL CHANGE ===
# Location of secrets.json file. In the default provided configuration it is not needed as the secrets are injected in the payload through environment variable,
# but if you want to use a dedicated secrets file instead it is possible
local_secrets_file = None    # example: "./config/secrets.json"

### 3.2 - Create Dask cluster

Create a new Dask cluster for the processor chosen in the previous step.   
The image used is adapted to your environment (local or cluster) and the tag specified in the previous step.   

**IMPORTANT:** For any **new** processor added, add the corresponding "case" in the cell below, following what is done for the existing ones.

In [ ]:
from resources.utils import *  
from resources.dask_clusters.dask_main_env import *
from rs_client.ogcapi.dpr_client import DprProcessor

# Init Dask cluster for chosen processor
match dpr_proc_radio.value:
    case "mockup":
        cluster_info = await init_dask_cluster_mockup(scale=1)
    case DprProcessor.S1L0.value | DprProcessor.S3L0.value:
        cluster_info = await init_dask_cluster_l0(
            image_tag=processor_image_tag,
            **dask_cluster_parameters,
        )
    case DprProcessor.S1ARD.value:
        cluster_info = await init_dask_cluster_s1ard(
            image_tag=processor_image_tag,
            **dask_cluster_parameters,
        )
    case DprProcessor.S3OLCI.value:
        cluster_info = await init_dask_cluster_s3olci(
            image_tag=processor_image_tag,
            **dask_cluster_parameters,
        )
    # === OPTIONAL CHANGE ===
    # Add here any new processor to the list

In [ ]:
# Init demo
init_demo()
# Reload the global vars again
from resources.utils import *

# Other imports
import os.path as osp
from resources.dpr_utils import DprDemo

## 4 - Run the processor <a class="anchor" id="step4"></a>

This part is the run of the processor with the test data you selected. Make sure to change the "payload_location" variable below to match the payload created at previous step. All other fields and parameters should be populated automatically, and two of them can be tuned easily (see optional changes). Once your configuration is set up in the first cell, simply run the second one without any change.   

Once you run the second cell, the processor will be triggered and run in background. The execution of the processor can vary a lot depending on which one is running and with which data, sometimes up to several hours. 

In [ ]:
# Init DPR processor demo
dpr = DprDemo(
    owner_id=OWNER_ID, 
    dpr_client=dpr_client,
    local_config_dir=f"./config_{dpr_proc_radio.value}"
)

await dpr.init(local_secrets_file=local_secrets_file)

# === MANDATORY CHANGE ===
# Location of the payload to use (relative to the "config" folder). Default value is an example payload.
payload_location = "s1/basic_payload.yaml"

# === OPTIONAL CHANGES ===
# Name of the local subfolder in which the log files will be uploaded. By default the name of the processor, but can be replaced with any string.
report_dir = dpr_proc_radio.value

print(f"=== PATH TO THE PAYLOAD ===\n./config_{dpr_proc_radio.value}/{payload_location}")
print(f"=== PATH TO THE REPORT DIRECTORY ===\n./reports/{report_dir}")

In [ ]:
# Arguments for the processor for this run
proc_args = {
    "process": dpr_proc_radio.value,               # Reference of the processor to run in rs-dpr-service from step 3.1
    "cluster_info": cluster_info,                  # Reference to the cluster created in step 3.2
    "payload_subpath": payload_location,           # Path to the payload file created at step 2
    "experimental_config": experimental_config,    # Experimental config set in step 3.1   
}

# Run the processor
proc_output_dir = osp.join(dpr.s3_output_dir, "dask_cluster", report_dir)
await dpr.run_process(
    **proc_args,
    s3_output_dir = proc_output_dir,
    s3_report_dir = osp.join(dpr.s3_report_dir, "dask_cluster", report_dir),
    # Payload env vars
    OUTPUT_DIR = proc_output_dir,
)

## 5 - Retrieve informations and shutdown cluster <a class="anchor" id="step5"></a>

### 5.1 - Tips to retrieve informations from processor's run

Wait for the process launched at step 3 to finished, either with a "error" or a "success" message. Below are a few indications of which informations are useful in each case and how to retrieve them.

**IF THE PROCESS FAILED:** first, identify why it failed. Usually there are two cases: a bug in the processor, or a problem in the runtime environment.    
To identify if the bug is coming from the processor, check if a log file has been generated, it contains the logs from the processor. If it contains error logs, it means they were generated by the processor and you are in case 1, but if the file doesn't exist or doesn't contain error logs you are most likely in case 2.

**Case 1:** If the bug is inside the processor, retrieve the **log file** created at the location `report_dir`. Make sure the bug is not related to a wrong test data set or payload that you can fix yourself. If it is not the case, open a ticket on the processor's repository with as much data as possible (log file, payload, configuration files, environment used).   
To identify more precisely a bug, you can also try to debug the processor yourself by running it using the `experimental_config` setup in local mode: for this, you need to run the local mode with a `localcluster` image for the processor in place of the regular rs-dpr-service image, and run it in debug mode.   

**Case 2:** If there is a problem with the runtime environment, usually a Dask cluster not running properly, check that all the needed images are up and running in the cluster (using k9s for example) and redo the procedure from the beginning if there is a problem. Sometimes the issues are external to the processor or this procedure, so reach out to the cluster administrator if something unusual is happening.   
For processors with a long execution time, random timeouts can cause the process to be shown as "failed" in the notebook. In such cases, the processor is often still running in the background, but it may be more complicated to retrieve log informations. You can check if the processor is still running by clicking on the Dask dashboard link given when the cluster was created, where you can see if it is still active or not. Do not hesitate to shutdown the Dask cluster and redo the procedure from the start if this happens.

**IF THE PROCESS SUCCEEDED:** here is a list of the useful metrics to retrieve, to make comparisons with other versions of the processor. See also [this Confluence page](https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/1.+Validation+reports) for detailed informations.   

- **Execution time:** shown in the output of the processor's triggering cell (step 4), just before the logs. Be careful as it is only displayed there, it is not printed in the log file.   
- **Max CPU consumption per worker:** to evaluate the processor's efficiency. Go in the Grafana dashboard: "Dashboards" (left menu) > "Kubernetes / Views / Pods" (in the list) > select the correct namespace in the top options (usually "dask-gateway") > look for the panel called "CPU Usage by container" > identify the "dask-worker" pod used by your processor and get the associated "Max" value
- **Max RAM consumption per worker:** same procedure as in previous point, but using the panel called "Memory Usage by container"
- **Max RAM consumption for the scheduler:** same procedure as in previous point, but with the "dask-scheduler"
- **List of generated files:** adapt and run the following command in a bash prompt, using the correct output subfolder (depends on your payload): `s3cmd ls s3://rs-dev-cluster-temp/prefect-share/users/YOUR-USERNAME/output/OUTPUT-SUBFOLDER -rH`
- **Size of generated files:** use the following command on the same path as for the previous one: `s3cmd du -H s3://rs-dev-cluster-temp/prefect-share/users/YOUR-USERNAME/output/l0/s3/S3_D_TDS_1/output/default/L0_Products`

**IN BOTH CASES:** Unless you are in the case 2 of a failed process (issue with the runtime environment), please **fill a report for your tests** in the DPRs integration results page in Confluence, available [**here**](https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/2.+DPR+integration+RESULTs). For a successful process, provide the informations detailed previously, and less importantly the payload used and the log file. For a failed process, detail the issue encountered, add the log file and the payload used, add a link to any issue opened on the source repository of the processor, and less importantly the informations usually given for a successful process.

### 5.2 - Shutdown Dask cluster (optional)

Shuting down the Dask cluster after usage is not mandatory (on the dev cluster the Dask clusters are automatically removed each night), but it can be useful in some cases. If you want to shutdown the Dask cluster, click on the link printed in the first cell of step 1.2 to access the *init_dask_cluster* notebook used to create the Dask cluster, and inside it run all the cells to the one that shutdowns it (the first cells will retrieve the existing active cluster).

### 5.3 - Useful links

Here is a list of links that can be useful on various steps of the integration:
- Full integration procedure: https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/1.+DPR+integration+PROCEDURE
- Old integration tutorial for S1L0 processor: https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/S1-L0+integration (exists also for S3L0 and S1ARD)
- Old integration results for S1L0 processor: https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/S1-L0+tests+-+V1.4.2 (exists also for other versions of the processor and S3L0)
- Confluence page explaining how to retrieve informations for the validation reports: https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/1.+Validation+reports
- Grafana dashboard on the dev cluster: https://monitoring.dev-rspy-ovh.esa-copernicus.eu/
- Documentation for LocalCluster mode and Local Files: https://github.com/RS-PYTHON/rs-dpr-service/blob/develop/rs_dpr_service/utils/settings.py, https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/1.+Validation+reports